## Recovering Correlation Energy: B atom

:::{panels} 
:column: col-12
Based on **Basis sets**
 Tricia D. Shepherd [![Orcid](../../images/orcid.png)](https://orcid.org/0000-0001-6512-8951), Ryan C. Fortenberry, Matthewy Kennedy, C. David Sheril, 2008-2019, The Psi4Education Developers &nbsp;&nbsp; License BSD-3
:::

At the Hartree Fock level of theory, each electron experiences an average potential field of all the other electrons. In essence, it is a "mean field" approach that neglects individual electron-electron interactions or "electron correlation". Thus, we define the difference between the self-consistent field energy and the exact energy as the correlation energy. Two fundamentally different approaches to account for electron correlation effects are available by selecting a Correlation method: Moller Plesset (MP) Perturbation theory and Coupled Cluster (CC) theory. 

:::{admonition} Exercise 1
:class: exercise 

* Calculate the HF energy for Boron using the 6-311+G** basis set, then determine the value of the correlation energy for boron assuming an "experimental" energy of -24.608 Hartrees ([Schaefer III, Henry F., and Frank E. Harris. "Electronic Structure of Atomic Boron." Physical Review 167.1 (1968): 67](https://journals.aps.org/pr/abstract/10.1103/PhysRev.167.67)).  
* Using the same basis set, perform an energy calculation with CISD and full CI. 
* Using the same basis set, perform an energy calculation with MP2, MP3 and MP5. 
* Using the same basis set, perform an energy calculation with CCSD and CCSD(T).

Determine the percentage of the correlation energy recovered for HF, MP2, MP5, CCSD, CCSD(T). 

Compare the performances of the different methods. In particular, you will see that some of the methods seem to overestimate the correlation recovered. How is it possible? (*Hint:* Think both about the theoretical aspects of the methods and *how* we are computing the recovered correlation energy).
:::

:::{admonition} CCSD energy/mp2 energy
:class: tip
You may recover the CCSD energy from the CCSD(T) calculation, but you will have to look at the output file or by calculating both. In any case, the calculations for a single atom are quick.  

The same is true for the MP2 energy which can be extracted from the MP5 calculation.
MP5 will require the use of the following options: `psi4.set_options({"reference" :"rohf", "qc_module":"detci"})` as Psi4 does not have MP5 implemented using a RHF reference. 
:::

In [2]:
import psi4
import py3Dmol
import pandas as pd
import matplotlib.pyplot as plt

import sys
sys.path.append("..")
from helpers import *

In [3]:
psi4.set_num_threads(2)
psi4.set_memory('2 GB')

2000000000

In [22]:
psi4.core.clean_options()

# define a single boron atom and set the correct multiplicity
boron_geometry = psi4.geometry("""
B
""")

psi4.set_options({'reference' : "UHF"})

# Perform single-point calculation with requested method/basis 

B_energy_hf  = psi4.energy('hf/6-311+G**', molecule=boron_geometry)
print(F"\n Hartree-Fock Energy: {B_energy_hf} Hartrees")


 Hartree-Fock Energy: -24.5303451727727 Hartrees


In [11]:
drawXYZ(boron_geometry)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [12]:
#Experimental energy of Boron
B_exact_e = -24.608


correlation = B_exact_e-B_energy_hf # Calculate the total amount of correlation energy that should be recovered using the B_exact_e

print(F"\n Electron correlation: {correlation} Hartrees")


 Electron correlation: -0.07765482722729544 Hartrees


In [21]:
# calculate energy for other methods
psi4.core.clean_options()
psi4.core.clean()
psi4.set_options({'reference' : "UHF"})
#MP2
e_mp2 = psi4.energy('mp2/6-311+G**', molecule=boron_geometry)
#MP3
e_mp3 = psi4.energy('mp3/6-311+G**', molecule=boron_geometry)
#CCSD
e_ccsd = psi4.energy('CCSD/6-311+G**', molecule=boron_geometry)
#CCSD(T)
e_ccsd_t = psi4.energy('CCSD(T)/6-311+G**', molecule=boron_geometry)
#MP5
psi4.core.clean_options()
psi4.core.clean()
psi4.set_options({"reference" :"rohf", 
                   "qc_module":"detci"})
e_mp5 = psi4.energy('mp5/6-311+G**', molecule=boron_geometry)
#CISD
e_cisd = psi4.energy('CISD/6-311+G**', molecule=boron_geometry)
#FCI
e_fci = psi4.energy('FCI/6-311+G**', molecule=boron_geometry)

print(F"\n MP2 Energy: {e_mp2} Hartrees")
print(F"\n MP3 Energy: {e_mp3} Hartrees")
print(F"\n MP5 Energy: {e_mp5} Hartrees")
print(F"\n CCSD Energy: {e_ccsd} Hartrees")
print(F"\n CCSD(T) Energy: {e_ccsd_t} Hartrees")
print(F"\n CISD Energy: {e_cisd} Hartrees")
print(F"\n FCI Energy: {e_fci} Hartrees")


 MP2 Energy: -24.586289575122024 Hartrees

 MP3 Energy: -24.60080134842056 Hartrees

 MP5 Energy: -24.60860981858082 Hartrees

 CCSD Energy: -24.60855241969005 Hartrees

 CCSD(T) Energy: -24.609669719516084 Hartrees

 CISD Energy: -24.60742517905764 Hartrees

 FCI Energy: -24.6100439750888 Hartrees


In [23]:
correlations = {}
correlations['hf']= 0
correlations['mp2']    = 100*(1-(B_exact_e-e_mp2)/correlation) # calculate the % of correlation energy recovered here
correlations['mp3']    = 100*(1-(B_exact_e-e_mp3)/correlation) # calculate the % of correlation energy recovered here
correlations['mp5']    = 100*(1-(B_exact_e-e_mp5)/correlation) # calculate the % of correlation energy recovered here
correlations['ccsd']   = 100*(1-(B_exact_e-e_ccsd)/correlation) # calculate the % of correlation energy recovered here
correlations['ccsd(t)'] = 100*(1-(B_exact_e-e_ccsd_t)/correlation) # calculate the % of correlation energy recovered here
correlations['cisd'] = 100*(1-(B_exact_e-e_cisd)/correlation) # calculate the % of correlation energy recovered here
correlations['fci'] = 100*(1-(B_exact_e-e_fci)/correlation) # calculate the % of correlation energy recovered here

In [24]:
pd.DataFrame.from_dict(correlations,orient='index',columns=['% correlation recovered'])

,% correlation recovered
hf,0.000000
mp2,72.042401
mp3,90.729937
mp5,100.785294
ccsd,100.711378
ccsd(t),102.150181
cisd,99.259774
fci,102.632129
